# Prenche colunas do csv

In [ ]:
import pandas as pd
from pathlib import Path
import bibtexparser

# Caminhos dos arquivos
csv_path = Path("accepted_articles.csv")
bib_path = Path("accepted_articles.bib")

# Carregar o CSV
df = pd.read_csv(csv_path)

# 1. Preencher células vazias nas colunas "size" e "model" com "n/a"
df["size"] = df["size"].fillna("n/a")
df["model"] = df["model"].fillna("n/a")

# 2. Associar DOIs com a chave do BibTeX
# Criar a coluna 'ref' se ainda não existir
if "ref" not in df.columns:
    df["ref"] = ""

# Carregar o arquivo .bib
with open(bib_path, "r", encoding="utf-8") as bib_file:
    bib_database = bibtexparser.load(bib_file)

# Criar dicionário DOI -> BibTeX key
doi_to_bibkey = {}
for entry in bib_database.entries:
    doi = entry.get("doi", "").strip().lower()
    if doi:
        doi_to_bibkey[doi] = entry["ID"]

# Preencher a coluna 'ref' usando os DOIs
for idx, row in df.iterrows():
    if row["ref"] == "" and pd.notna(row["doi"]):
        doi_key = row["doi"].strip().lower()
        matched_key = doi_to_bibkey.get(doi_key)
        if matched_key:
            df.at[idx, "ref"] = matched_key

# Salvar o CSV atualizado
df.to_csv(csv_path, index=False)

df.head()

,title,year,source,selection_criteria,status,comments,display,input,interaction,collaboration,model,size,doi,ref
0,Trains of thought on the tabletop: visualizing...,2013,Springer Link,Study describing digital tabletop application ...,Accepted,DiamondTouch,Front-Projected,Electrical-Based,"Multi-touch, Touch",Colaboração presencial,DiamondTouch,42”,10.1007/s00779-013-0726-3,Afonso_Jaco_2013
1,Sketch-Play-Learn - An Augmented Paper Based E...,2015,ACM Digital Library,Study describing digital tabletop techniques,Accepted,NaN,Rear-Projected,"Tangible-Tokens, Vision-Based",Tangibility,Colaboração presencial,n/a,n/a,10.1145/2757226.2764558,Agarwal_2015
2,ChemicAble: Tangible Interaction Approach for ...,2013,ACM Digital Library,Study describing digital tabletop application ...,Accepted,NaN,Rear-Projected,"Tangible-Tokens, Vision-Based",Tangibility,Colaboração presencial,n/a,n/a,10.1145/2525194.2525211,Agrawal_2013
3,A slim tabletop interface based on high resolu...,2010,Scopus,Study describing digital tabletop techniques,Accepted,NaN,Embedded,"Sensor-Assisted-Vision-Based, Vision-Based","Multi-touch, Touch",Colaboração presencial,n/a,n/a,10.1145/1936652.1936696,Ahn_2010
4,A slim hybrid multi-touch tabletop interface w...,2012,Scopus,Study describing digital tabletop techniques,Accepted,NaN,Embedded,"Sensor-Assisted-Vision-Based, Vision-Based","Multi-touch, Touch",Colaboração presencial,n/a,"47""",10.1109/ICCE.2012.6161836,Ahn_2012


# Cria a tabela latex

In [ ]:
import pandas as pd
from pathlib import Path

# Caminho para o CSV
csv_path = Path("accepted_articles.csv")

# Carregar os dados
df = pd.read_csv(csv_path)

# Selecionar e renomear as colunas relevantes
df_table = df[["ref", "display", "input", "interaction", "collaboration", "model", "size"]].copy()
df_table.columns = ["Referência", "Display", "Input", "Interação", "Colaboração", "Modelo", "Tamanho"]

# Função para escapar caracteres especiais no LaTeX
def latex_escape(text):
    if pd.isna(text):
        return "n/a"
    return str(text).replace('&', r'\&').replace('%', r'\%').replace('_', r'\_').replace('$', r'\$')

# Escapar todas as colunas, exceto "Referência"
for col in df_table.columns:
    if col != "Referência":
        df_table[col] = df_table[col].apply(latex_escape)

# Gerar o conteúdo LaTeX
latex_lines = []
latex_lines.append("\\begin{table*}%")
latex_lines.append("\\caption{Artigos aceitos e suas características extraídas.\\label{tab:artigos_classificados}}")
latex_lines.append("\\begin{tabular*}{\\textwidth}{@{\\extracolsep\\fill}lllllll@{}}\\toprule")
latex_lines.append("\\textbf{Referência} & \\textbf{Display} & \\textbf{Input} & \\textbf{Interação} & \\textbf{Colaboração} & \\textbf{Modelo} & \\textbf{Tamanho} \\\\")
latex_lines.append("\\midrule")

# Adicionar linhas da tabela
for _, row in df_table.iterrows():
    row["Referência"] = f"\\cite{{{row['Referência']}}}"
    line = " & ".join(row.values) + " \\\\"
    latex_lines.append(line)

latex_lines.append("\\bottomrule")
latex_lines.append("\\end{tabular*}")
latex_lines.append("\\end{table*}")

# Salvar como .tex
output_path = Path("tabela_artigos.tex")
with open(output_path, "w", encoding="utf-8") as f:
    f.write("\n".join(latex_lines))

# Mostrar as primeiras linhas da tabela LaTeX gerada
latex_preview = "\n".join(latex_lines[:15])
latex_preview


<>:18: SyntaxWarning: invalid escape sequence '\&'
<>:18: SyntaxWarning: invalid escape sequence '\%'
<>:18: SyntaxWarning: invalid escape sequence '\_'
<>:18: SyntaxWarning: invalid escape sequence '\$'
<>:18: SyntaxWarning: invalid escape sequence '\&'
<>:18: SyntaxWarning: invalid escape sequence '\%'
<>:18: SyntaxWarning: invalid escape sequence '\_'
<>:18: SyntaxWarning: invalid escape sequence '\$'
C:\Users\gugli\AppData\Local\Temp\ipykernel_10856\3443475230.py:18: SyntaxWarning: invalid escape sequence '\&'
  return str(text).replace('&', '\&').replace('%', '\%').replace('_', '\_').replace('$', '\$')
C:\Users\gugli\AppData\Local\Temp\ipykernel_10856\3443475230.py:18: SyntaxWarning: invalid escape sequence '\%'
  return str(text).replace('&', '\&').replace('%', '\%').replace('_', '\_').replace('$', '\$')
C:\Users\gugli\AppData\Local\Temp\ipykernel_10856\3443475230.py:18: SyntaxWarning: invalid escape sequence '\_'
  return str(text).replace('&', '\&').replace('%', '\%').replace('

'\\begin{table*}%\n\\caption{Artigos aceitos e suas características extraídas.\\label{tab:artigos_classificados}}\n\\begin{tabular*}{\\textwidth}{@{\\extracolsep\\fill}lllllll@{}}\\toprule\n\\textbf{Referência} & \\textbf{Display} & \\textbf{Input} & \\textbf{Interação} & \\textbf{Colaboração} & \\textbf{Modelo} & \\textbf{Tamanho} \\\\\n\\midrule\n\\cite{Shen_2002} & Front-Projected & Pen & Tangibility & Colaboração presencial & n/a & n/a \\\\\n\\cite{Vernier_2002} & Front-Projected & Pen, Sound-Based & Tangibility & Colaboração presencial & n/a & n/a \\\\\n\\cite{Shen_2003} & Front-Projected & Electrical-Based & Multi-touch, Touch & Colaboração presencial & DiamondTouch & 42” \\\\\n\\cite{Shen_2003} & Front-Projected & Electrical-Based & Multi-touch, Touch & Colaboração presencial & DiamondTouch & 42” \\\\\n\\cite{Moghaddam_2004} & Front-Projected & Pen & Tangibility & Colaboração presencial & n/a & n/a \\\\\n\\cite{Morris_2004} & Front-Projected & Electrical-Based & Multi-touch, Tou